In [6]:
import numpy as np
from scipy.spatial import cKDTree


# Read biochar

In [51]:

structure_name =  input("structure_name")

date =structure_name + '_slab'

structure_name C700_000


In [52]:
#toggle here
fname = '../../structure_relax/fixed_COM/' + structure_name +'/results/polish/polish.data'
#fname = '../../structure_relax/' + structure_name + '.data'

# Reading data files

In [53]:
def read_lammps_data_charge_id_type_q_xyz(fname):
    with open(fname, "r") as f:
        lines = f.readlines()

    box = {}
    masses_lines = []
    atoms = []

    def is_section_header(s):
        s = s.strip()
        return s in {"Masses", "Atoms", "Bonds", "Angles", "Dihedrals", "Impropers", "Velocities",
                     "Pair Coeffs", "Bond Coeffs", "Angle Coeffs", "Dihedral Coeffs", "Improper Coeffs"}

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if "xlo xhi" in line:
            box["xlo"], box["xhi"] = map(float, line.split()[:2])
        elif "ylo yhi" in line:
            box["ylo"], box["yhi"] = map(float, line.split()[:2])
        elif "zlo zhi" in line:
            box["zlo"], box["zhi"] = map(float, line.split()[:2])

        if line == "Masses":
            i += 2
            while i < len(lines) and lines[i].strip() and not is_section_header(lines[i]):
                masses_lines.append(lines[i].rstrip("\n"))
                i += 1

        if line.startswith("Atoms"):
            i += 2
            while i < len(lines) and lines[i].strip() and not is_section_header(lines[i]):
                parts = lines[i].split()
                atoms.append([int(parts[0]), int(parts[1]), float(parts[2]),
                              float(parts[3]), float(parts[4]), float(parts[5])])
                i += 1

        i += 1

    return np.array(atoms, dtype=float), box, masses_lines


atoms, box, masses_lines = read_lammps_data_charge_id_type_q_xyz(fname)

In [54]:
box

{'xlo': -27.9674369314583,
 'xhi': 28.176787931458296,
 'ylo': -28.178633931458297,
 'yhi': 27.965590931458298,
 'zlo': -28.072112815323518,
 'zhi': 28.072112815323518}

In [55]:
def shift_system_to_origin(atoms, box, shift_z_to_min=True):
    """
    Shift coordinates so that box corner is at (0,0,0).
    Optionally also shift z so the minimum atom z is at 0 (useful for slabs).
    atoms columns: [id, type, q, x, y, z]
    """
    atoms = atoms.copy()
    box = dict(box)

    # Shift by box lower bounds so that xlo=ylo=zlo=0
    dx = box["xlo"]
    dy = box["ylo"]
    dz = box["zlo"]

    atoms[:, 3] -= dx
    atoms[:, 4] -= dy
    atoms[:, 5] -= dz

    # Update box to 0..L
    box["xhi"] -= box["xlo"]
    box["yhi"] -= box["ylo"]
    box["zhi"] -= box["zlo"]
    box["xlo"] = 0.0
    box["ylo"] = 0.0
    box["zlo"] = 0.0

    # Optional: anchor slab bottom exactly at z=0 based on atoms
    if shift_z_to_min:
        zmin = atoms[:, 5].min()
        atoms[:, 5] -= zmin
        box["zhi"] -= zmin  # keep containment consistent

    return atoms, box

In [56]:
atoms, box = shift_system_to_origin(atoms, box, shift_z_to_min=False)
box

{'xlo': 0.0,
 'xhi': 56.144224862916595,
 'ylo': 0.0,
 'yhi': 56.144224862916595,
 'zlo': 0.0,
 'zhi': 56.144225630647036}

# Create vacuum gap

In [57]:
import numpy as np

# ---------------------------
# Geometry / replication
# ---------------------------
REPX, REPY = 1, 1      # replicate in the two in-plane directions
VACUUM_GAP = 100.0      # Å vacuum along surface normal

# ---------------------------
# ReaxFF type IDs for C H N O order
# pair_coeff * * ffield.reax.chon C H N O
# ---------------------------
TYPE_C, TYPE_H, TYPE_N, TYPE_O = 1, 2, 3, 4


def replicate_in_plane(atoms, box, surface_normal, rep1=1, rep2=1):
    """
    Replicate the structure only in the two directions perpendicular
    to the chosen surface normal.

    Assumes atom columns:
      id, mol, type, x, y, z, ...
    with x/y/z at cols 3/4/5.
    """
    atoms = atoms.copy()

    axis_to_col = {"x": 3, "y": 4, "z": 5}

    normal_axis = surface_normal[1]   # 'x', 'y', or 'z'
    in_plane_axes = [ax for ax in ("x", "y", "z") if ax != normal_axis]

    ax1, ax2 = in_plane_axes
    col1, col2 = axis_to_col[ax1], axis_to_col[ax2]

    L1 = box[f"{ax1}hi"] - box[f"{ax1}lo"]
    L2 = box[f"{ax2}hi"] - box[f"{ax2}lo"]

    replicated = []
    next_id = 1

    for i in range(rep1):
        for j in range(rep2):
            block = atoms.copy()
            block[:, col1] += i * L1
            block[:, col2] += j * L2
            block[:, 0] = np.arange(next_id, next_id + len(block))
            next_id += len(block)
            replicated.append(block)

    atoms_rep = np.vstack(replicated)

    new_box = dict(box)
    new_box[f"{ax1}lo"] = 0.0
    new_box[f"{ax1}hi"] = rep1 * L1
    new_box[f"{ax2}lo"] = 0.0
    new_box[f"{ax2}hi"] = rep2 * L2

    return atoms_rep, new_box


def make_slab_with_vacuum(atoms, box, vacuum_gap, surface_normal="+z"):
    """
    Shift and optionally flip the structure so that vacuum is added
    along the requested surface normal.

    surface_normal: one of {"+x", "-x", "+y", "-y", "+z", "-z"}

    Returns
    -------
    atoms_new, new_box, slab_thickness, final_hi
    """
    atoms_new = atoms.copy()
    new_box = dict(box)

    axis_to_col = {"x": 3, "y": 4, "z": 5}

    if surface_normal not in {"+x", "-x", "+y", "-y", "+z", "-z"}:
        raise ValueError("surface_normal must be one of '+x', '-x', '+y', '-y', '+z', '-z'")

    sign = surface_normal[0]
    axis = surface_normal[1]
    col = axis_to_col[axis]

    coords = atoms_new[:, col]

    if sign == "+":
        # expose the +axis face; shift slab so its low edge is at 0
        atoms_new[:, col] -= coords.min()
    else:
        # expose the -axis face by flipping that coordinate
        atoms_new[:, col] = coords.max() - coords
        atoms_new[:, col] -= atoms_new[:, col].min()

    slab_thickness = atoms_new[:, col].max()
    final_hi = slab_thickness + vacuum_gap

    new_box[f"{axis}lo"] = 0.0
    new_box[f"{axis}hi"] = float(final_hi)

    return atoms_new, new_box, float(slab_thickness), float(final_hi)


def build_surface_slab(atoms, box, surface_normal, repx=1, repy=1, vacuum_gap=100.0):
    """
    Full workflow:
      1. replicate in the two in-plane directions
      2. shift/flip structure
      3. add vacuum along chosen normal
    """
    atoms_rep, box_rep = replicate_in_plane(
        atoms, box, surface_normal, rep1=repx, rep2=repy
    )

    slab_atoms, slab_box, slab_thickness, final_hi = make_slab_with_vacuum(
        atoms_rep, box_rep, vacuum_gap=vacuum_gap, surface_normal=surface_normal
    )

    return slab_atoms, slab_box, slab_thickness, final_hi

In [58]:
surface_normals = ["+x", "-x", "+y", "-y", "+z", "-z"]

all_slabs = {}
for normal in surface_normals:
    slab_atoms, slab_box, thickness, hi = build_surface_slab(
        atoms,
        box,
        surface_normal=normal,
        repx=REPX,
        repy=REPY,
        vacuum_gap=VACUUM_GAP
    )
    all_slabs[normal] = {
        "atoms": slab_atoms,
        "box": slab_box,
        "thickness": thickness,
        "hi": hi
    }

In [59]:
all_slabs

{'+x': {'atoms': array([[ 1.00000000e+00,  2.00000000e+00,  3.04555586e-02,
           1.73174755e+01,  3.44214413e+00,  5.75402682e+00],
         [ 2.00000000e+00,  2.00000000e+00,  2.67671230e-02,
           1.52015834e+01,  4.81490946e+00,  5.74439334e+00],
         [ 3.00000000e+00,  1.00000000e+00, -1.69194311e-02,
           1.77238299e+01,  4.21417281e+00,  6.38935993e+00],
         ...,
         [ 1.05800000e+04,  1.00000000e+00, -6.71010946e-02,
           1.36603183e+01,  4.99082662e+01,  5.52323317e+01],
         [ 1.05810000e+04,  2.00000000e+00,  6.85499688e-02,
           1.40983076e+01,  5.07411390e+01,  5.46426479e+01],
         [ 1.05820000e+04,  2.00000000e+00,  2.57899553e-02,
           1.62561393e+01,  4.95366815e+01,  5.45230104e+01]],
        shape=(10582, 6)),
  'box': {'xlo': 0.0,
   'xhi': 153.15930608441246,
   'ylo': 0.0,
   'yhi': 56.144224862916595,
   'zlo': 0.0,
   'zhi': 56.144225630647036},
  'thickness': 53.159306084412464,
  'hi': 153.15930608441246}

In [60]:
import os
import numpy as np


def write_lammps_data_charge_id_type_q_xyz(fname, atoms, box, masses_lines,
                                           header="LAMMPS data"):
    """
    Write a LAMMPS data file with atom format:

        id  type  q  x  y  z

    Parameters
    ----------
    fname : str
        Output filename
    atoms : np.ndarray
        Array with columns [id, type, q, x, y, z]
    box : dict
        Must contain xlo, xhi, ylo, yhi, zlo, zhi
    masses_lines : list[str]
        Lines for the Masses section, e.g.:
            ["1 12.011", "2 1.008", "3 14.007", "4 15.999"]
    header : str
        Header line for data file
    """
    atoms = atoms.copy()
    atoms = atoms[np.argsort(atoms[:, 0])]   # sort by atom ID

    natoms = atoms.shape[0]
    ntypes = int(atoms[:, 1].max())

    with open(fname, "w") as f:
        f.write(f"{header}\n\n")
        f.write(f"{natoms} atoms\n")
        f.write(f"{ntypes} atom types\n\n")

        f.write(f"{box['xlo']:.6f} {box['xhi']:.6f} xlo xhi\n")
        f.write(f"{box['ylo']:.6f} {box['yhi']:.6f} ylo yhi\n")
        f.write(f"{box['zlo']:.6f} {box['zhi']:.6f} zlo zhi\n\n")

        f.write("Masses\n\n")
        for ml in masses_lines:
            f.write(ml + "\n")
        f.write("\n")

        f.write("Atoms # charge\n\n")
        for a in atoms:
            atom_id = int(a[0])
            atom_type = int(a[1])
            q, x, y, z = a[2], a[3], a[4], a[5]
            f.write(f"{atom_id} {atom_type} {q:.6f} {x:.6f} {y:.6f} {z:.6f}\n")
        f.write("\n")

In [61]:
import os

surface_normals = ["+x", "-x", "+y", "-y", "+z", "-z"]

label_map = {
    "+x": "px",
    "-x": "nx",
    "+y": "py",
    "-y": "ny",
    "+z": "pz",
    "-z": "nz",
}

masses_lines = [
    f"{TYPE_C} 12.011",
    f"{TYPE_H} 1.008",
    f"{TYPE_N} 14.007",
    f"{TYPE_O} 15.999",
]

# ---------------------------
# output directory with structure name
# ---------------------------
base_outdir = "slab_faces"
outdir = os.path.join(base_outdir, structure_name)
os.makedirs(outdir, exist_ok=True)

all_slabs = {}

for normal in surface_normals:
    slab_atoms, slab_box, thickness, hi = build_surface_slab(
        atoms,
        box,
        surface_normal=normal,
        repx=REPX,
        repy=REPY,
        vacuum_gap=VACUUM_GAP
    )

    tag = label_map[normal]

    # option A: clean filenames inside structure folder
    fname = os.path.join(outdir, f"slab_{tag}.data")

    # option B (if you prefer everything self-contained in filename):
    # fname = os.path.join(outdir, f"{structure_name}_slab_{tag}.data")

    write_lammps_data_charge_id_type_q_xyz(
        fname=fname,
        atoms=slab_atoms,
        box=slab_box,
        masses_lines=masses_lines,
        header=f"{structure_name} slab {normal}"
    )

    all_slabs[normal] = {
        "atoms": slab_atoms,
        "box": slab_box,
        "thickness": thickness,
        "fname": fname,
    }

    print(f"Wrote {fname}")

Wrote slab_faces/C700_000/slab_px.data
Wrote slab_faces/C700_000/slab_nx.data
Wrote slab_faces/C700_000/slab_py.data
Wrote slab_faces/C700_000/slab_ny.data
Wrote slab_faces/C700_000/slab_pz.data
Wrote slab_faces/C700_000/slab_nz.data


# Build inputs

In [62]:
NBIO = atoms.shape[0]

#ZTOP = float(atoms_slab[:, 5].max())

print(NBIO)
#print(ZTOP)


10582


In [63]:
import os

def axis_settings_from_face(surface_normal):
    axis = surface_normal[1]

    if axis == "x":
        boundary = "f p p"
        fixvar_line = "variable        xfix equal 10.0"
        region_line = "region          bottom block EDGE ${xfix} EDGE EDGE EDGE EDGE units box"
        wall_line   = "fix             zwall mobile wall/lj126 xhi EDGE 1.0 2.5 2.8 units box"

    elif axis == "y":
        boundary = "p f p"
        fixvar_line = "variable        yfix equal 10.0"
        region_line = "region          bottom block EDGE EDGE EDGE ${yfix} EDGE EDGE units box"
        wall_line   = "fix             zwall mobile wall/lj126 yhi EDGE 1.0 2.5 2.8 units box"

    elif axis == "z":
        boundary = "p p f"
        fixvar_line = "variable        zfix equal 10.0"
        region_line = "region          bottom block EDGE EDGE EDGE EDGE EDGE ${zfix} units box"
        wall_line   = "fix             zwall mobile wall/lj126 zhi EDGE 1.0 2.5 2.8 units box"

    else:
        raise ValueError(f"Unsupported axis: {axis}")

    return {
        "axis": axis,
        "boundary": boundary,
        "fixvar_line": fixvar_line,
        "region_line": region_line,
        "wall_line": wall_line,
    }


def make_slab_input_text(surface_normal, data_filename, annealed_data_filename,
                         restart_prefix, traj_filename, species_filename,
                         bonds_filename, nbio):
    """
    Render a face-specific slab anneal input.
    """
    s = axis_settings_from_face(surface_normal)

    slab_in = f"""# ------------------------------------------------------------
# slab_anneal_{surface_normal}
# Biochar slab anneal (ReaxFF CHON) with soft wall
# ------------------------------------------------------------
log             lammps_run.log append
units           real
atom_style      charge
boundary        {s['boundary']}
newton          on

# ---------- Read system ----------
read_data       {data_filename}
restart         10000 {restart_prefix}.restart.*

# ---------- ReaxFF ----------
pair_style      reaxff lmp_control safezone 50 mincap 1000
pair_coeff      * * ffield.reax.chon C H N O

neighbor        2.5 bin
neigh_modify    every 1 delay 0 check yes

# ---------- Charge equilibration ----------
fix             qeq all qeq/reaxff 1 0.0 10.0 1.0e-8 reaxff maxiter 2000
compute         reaxatm all reaxff/atom

# ---------- Groups ----------
variable        NBIO equal {nbio}
group           char id <= ${{NBIO}}

# ---------- Freeze bottom 1 nm ----------
{s['fixvar_line']}
{s['region_line']}
group           rbottom region bottom
group           fixed intersect char rbottom
group           mobile subtract all fixed

velocity        fixed set 0.0 0.0 0.0
fix             freeze fixed setforce 0.0 0.0 0.0

# Remove net drift/rotation from mobile slab at startup
velocity        mobile zero linear
velocity        mobile zero angular

# ---------- Temperature compute ----------
compute         Tmobile mobile temp

# ---------- ReaxFF chemistry diagnostics ----------
fix             fspec all reaxff/species 1000 1 1000 {species_filename}
fix             fbond all reaxff/bonds 5000 {bonds_filename}

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------
thermo          1000
thermo_style    custom step time c_Tmobile etotal pe ke fmax fnorm
thermo_modify   temp Tmobile flush yes

dump            dtraj all custom 5000 {traj_filename} id type x y z ix iy iz q c_reaxatm[*]
dump_modify     dtraj element C H N O sort id flush yes append yes

# ------------------------------------------------------------
# RUN PLAN
# ------------------------------------------------------------
reset_timestep  0

# Conservative minimization to prevent first-step blowups
min_style       cg
min_modify      dmax 0.02
minimize        1.0e-8 1.0e-8 10000 100000

# ------------------------------------------------------------
# Soft wall at high edge of nonperiodic axis
# ------------------------------------------------------------
{s['wall_line']}

# ------------------------------------------------------------
# Stage 0: ultra-gentle relaxation
# ------------------------------------------------------------
timestep        0.03
velocity        mobile create 298.0 43 mom yes rot yes dist gaussian

fix             int1 mobile nve/limit 0.05
fix             th1  mobile langevin 298.0 298.0 50.0 43 zero yes
fix_modify      th1 temp Tmobile
run             5000
unfix           th1
unfix           int1

# ------------------------------------------------------------
# Stage 0b: still gentle, slightly looser cap
# ------------------------------------------------------------
fix             int0b mobile nve/limit 0.08
fix             th0b  mobile langevin 298.0 298.0 50.0 84 zero yes
fix_modify      th0b temp Tmobile
run             10000
unfix           th0b
unfix           int0b

# ------------------------------------------------------------
# Stage 1: gentle NVT relaxation at 298 K
# ------------------------------------------------------------
fix             nvt_relax mobile nvt temp 298.0 298.0 100.0
fix_modify      nvt_relax temp Tmobile
run             400000
unfix           nvt_relax

# ------------------------------------------------------------
# Stage 2: heat 298 -> 500 K at 100 K/ps
# ------------------------------------------------------------
fix             nvt_heat mobile nvt temp 298.0 500.0 100.0
fix_modify      nvt_heat temp Tmobile
run             40400
unfix           nvt_heat

# ------------------------------------------------------------
# Stage 3: hold at 500 K for 5 ps
# ------------------------------------------------------------
fix             nvt_hold mobile nvt temp 500.0 500.0 100.0
fix_modify      nvt_hold temp Tmobile
run             100000
unfix           nvt_hold

# ------------------------------------------------------------
# Stage 4: cool 500 -> 298 K at 5 K/ps
# ------------------------------------------------------------
fix             nvt_cool mobile nvt temp 500.0 298.0 100.0
fix_modify      nvt_cool temp Tmobile
run             808000
unfix           nvt_cool

# ------------------------------------------------------------
# Stage 5: final equilibration at 298 K
# ------------------------------------------------------------
fix             nvt_final mobile nvt temp 298.0 298.0 100.0
fix_modify      nvt_final temp Tmobile
run             500000
timestep        0.05
run             500000
timestep        0.07
run             1500000
unfix           nvt_final

# ------------------------------------------------------------
# End / save
# ------------------------------------------------------------
unfix           zwall

write_data      {annealed_data_filename}
write_restart   {restart_prefix}_annealed_last.restart
"""
    return slab_in

In [64]:
for normal in surface_normals:
    tag = label_map[normal]
    face_dir = os.path.join(base_outdir, structure_name, tag)
    os.makedirs(face_dir+'/inputs', exist_ok=True)
    os.makedirs(face_dir+'/logs', exist_ok=True)
    os.makedirs(face_dir+'/latest', exist_ok=True)
    os.makedirs(face_dir+'/results', exist_ok=True)

    data_path = os.path.join(face_dir+'/inputs', f"slab_{tag}.data")
    in_path   = os.path.join(face_dir+'/inputs', f"slab_anneal_{tag}.in")

    # write the .data file there too
    write_lammps_data_charge_id_type_q_xyz(
        fname=data_path,
        atoms=all_slabs[normal]["atoms"],
        box=all_slabs[normal]["box"],
        masses_lines=masses_lines,
        header=f"{structure_name} slab {normal}"
    )

    input_text = make_slab_input_text(
        surface_normal=normal,
        data_filename=f"slab_{tag}.data",
        annealed_data_filename=f"slab_{tag}_annealed.data",
        restart_prefix=f"slab_{tag}",
        traj_filename=f"slab_{tag}.lammpstrj",
        species_filename=f"species_{tag}.out",
        bonds_filename=f"bonds_{tag}.reaxff",
        nbio=NBIO,
    )

    with open(in_path, "w") as f:
        f.write(input_text)

# build files


In [65]:
ff_CHNO = "Reactive MD-force field: Combustion C/H/O force field + atom type N(May11, 2018)\n\
40 ! Number of general parameters\n\
  50.0000 !p(boc1)\n\
   9.5469 !p(boc2)\n\
  26.5405 !p(coa2)\n\
   0.6863 !p(trip4)\n\
   2.7295 !p(trip3)\n\
  70.0000 !kc2\n\
   1.0588 !p(ovun6)\n\
   4.1262 !p(trip2)\n\
  12.1176 !p(ovun7)\n\
  13.3056 !p(ovun8)\n\
 -68.9784 !p(trip1)\n\
   0.0000 !Lower Taper-radius (swa)\n\
  10.0000 !Upper Taper-radius (swb)\n\
   0.0000 !not used\n\
  33.8667 !p(val7)\n\
   6.0891 !p(lp1)\n\
   1.0563 !p(val9)\n\
   2.0384 !p(val10)\n\
   6.1431 !not used\n\
   6.9290 !p(pen2)\n\
   0.3989 !p(pen3)\n\
   3.9954 !p(pen4)\n\
   0.0000 !not used\n\
   5.7796 !p(tor2)\n\
  10.0000 !p(tor3)\n\
   1.9487 !p(tor4)\n\
   0.0000 !not used\n\
   2.1645 !p(cot2)\n\
   1.5591 !p(vdW1)\n\
   0.1000 !Cutoff for bond order*100 (cutoff)\n\
   2.1365 !p(coa4)\n\
   0.6991 !p(ovun4)\n\
  50.0000 !p(ovun3)\n\
   1.8512 !p(val8)\n\
   0.0000 !not used\n\
   0.0000 !not used\n\
   0.0000 !not used\n\
   1.0000 !not used\n\
   2.6962 !p(coa3)\n\
   2.0000 !triple bond on/off (0 for CO, 1 for all, 2 for CO and N2)\n\
4   ! Nr of atoms; atomID;ro(sigma); Val;atom mass;Rvdw;Dij;gamma\n\
      	 alfa;gamma(w);Val(angle);p(ovun5);n.u.;chiEEM;etaEEM;n.u.\n\
       	 ro(pipi);p(lp2);Heat\n\
  	 increment;p(boc4);p(boc3);p(boc5),n.u.;n.u.\n\
C    1.3674   4.0000  12.0000   2.0453    0.1444   0.9500   1.1706   4.0000\n\
     9.0000   1.5000   4.0000  27.5134   79.5548   5.0191   7.0500   0.0000\n\
     1.1168   0.0000      NaN  14.2732   24.4406   6.7313   0.8563   0.0000\n\
    -4.1021   5.0000   1.0564   4.0000    2.9663   0.0000   0.0000   0.0000\n\
H    0.9479   1.0000   1.0080   1.1364    0.0232   0.9900  -0.1000   1.0000\n\
     9.0643   4.7746   1.0000   0.0000  121.1250   4.7757   9.7732   1.0000\n\
    -0.1000   0.0000      NaN   2.5194    2.3785   0.2223   1.0698   0.0000\n\
   -15.7683   2.1488   1.0338   1.0000    2.8793   0.0000   0.0000   0.0000\n\
O    1.1939   2.0000  15.9990   1.9289    0.1201   0.9900   1.0981   6.0000\n\
    10.4842   8.2916   4.0000  28.8967  116.0768   7.9703   7.0500   2.0000\n\
     1.0479  20.0000      NaN  10.0338    2.2024   0.9942   0.9745   0.0000\n\
    -3.6141   2.7025   1.0493   4.0000    2.9225   0.0000   0.0000   0.0000\n\
N    1.3638   3.0000  14.0000 	1.7000    0.0967   0.8537   1.1943   5.0000\n\
     9.8544  10.4284   4.0000  41.8891  100.0000   7.7391   7.5000   2.0000\n\
     1.0200   0.0700      NaN   1.5271    2.9480   2.6234   0.9745   0.0000\n\
    -5.6116   2.0047   1.0183 	4.0000 	  2.5196   0.0000   0.0000   0.0000\n\
10   ! Nr of bonds; at1;at2;De(sigma);De(pi);De(pipi);p(be1);p(b\n\
       	  p(be2);p(bo3);p(bo4);n.u.;p(bo1);p(bo2)\n\
1  1   80.8865  107.9944  52.0636   0.5218  -0.3636  1.0000  34.9876  0.7769\n\
        6.1244   -0.1693   8.0804   1.0000  -0.0586  8.1850   1.0000  0.0000\n\
1  2  179.5195    0.0000   0.0000  -0.5242   0.0000  1.0000   6.0000  0.7187\n\
        5.4740    1.0000   0.0000   1.0000  -0.1144  6.7029   0.0000  0.0000\n\
2  2  113.9232    0.0000   0.0000  -0.5971   0.0000  1.0000   6.0000  0.9093\n\
        1.7152    1.0000   0.0000   1.0000  -0.0450  6.0710   0.0000  0.0000\n\
1  3  136.4945  164.1201   5.5000  -0.9159  -0.1075  1.0000  10.6519  0.8644\n\
        0.6858   -0.4602   9.5754   1.0000  -0.1745  4.5987   0.0000  0.0000\n\
3  3  148.0798  155.2406  20.1160  -1.0000  -0.1254  1.0000  33.0027  0.7790\n\
        0.7673   -0.1697   7.0028   1.0000  -0.1300  5.1959   1.0000  0.0000\n\
2  3  169.1351    0.0000   0.0000  -0.8810   0.0000  1.0000   6.0000  0.5757\n\
        1.5482 	  1.0000   0.0000   1.0000  -0.1788  4.6622   0.0000  0.0000\n\
1  4  146.4220  161.9411  83.1445  -0.0673  -0.7385  1.0000  20.5574  0.3439\n\
        1.1554   -0.7615   6.3243   1.0000  -0.1692  5.3062   1.0000  0.0000\n\
2  4  131.9942	  0.0000   0.0000  -0.2031   0.0000  1.0000   4.0000  0.4507\n\
       10.2925   -0.3653   0.0000   1.0000  -0.0527  8.0000   0.0000  0.0000\n\
3  4   78.7524  155.4183 100.4654   1.0000  -1.0000  1.0000  40.0000  0.1723\n\
        0.1607   -0.5703   5.5634   1.0000  -0.1969  4.9725   1.0000  0.0000\n\
4  4   81.3043   99.0989 144.9704  -0.6110  -0.7864  1.0000   5.0000  0.1000\n\
 	1.0202   -0.1368   8.0395   1.0000  -0.1463  3.8325   1.0000  0.0000 \n\
6     ! Nr of off-diagonal terms. at1;at2;Dij;RvdW;alfa;ro(sigma);r\n\
 1 2  0.1253  1.5717   9.9736  1.2057  -1.0000  -1.0000\n\
 2 3  0.1125  1.6311   8.7528  1.0929  -1.0000  -1.0000\n\
 1 3  0.0953  1.7397   8.8986  1.4256   1.1067   1.1265\n\
 1 4  0.1425  1.8737  10.3522  1.4256   1.3259   1.2082\n\
 2 4  0.0660  1.5027   8.8662  1.0548  -1.0000  -1.0000\n\
 3 4  0.1263  1.5263  10.0075  1.3841   1.2535   1.0000\n\
41    ! Nr of angles. at1;at2;at3;Thetao,o;p(val1);p(val2);p(coa1);\n\
 1 1 1  76.1370  34.6920  1.1328    0.0000  0.0050    0.3556  1.8065\n\
 1 1 2  68.0572   9.9461  4.7000    0.0000  0.4566    0.0000  1.8532\n\
 2 1 2  65.6815  35.0000  1.8622    0.0000  0.0490    0.0000  1.0937\n\
 1 2 2   0.0000   4.0000  7.2043    0.0000  0.0000    0.0000  1.0728\n\
 1 2 1   0.0000   3.4110  7.7350    0.0000  0.0000    0.0000  1.0400\n\
 2 2 2   0.0000  30.0000  5.6235    0.0000  0.0000    0.0000  1.0400\n\
 1 1 3  78.3624  13.0773  9.0480    0.0000  0.1270   52.1129  2.3964\n\
 3 1 3  76.7101  24.3833  5.8613  -21.8559  2.6395  -32.6534  3.6179\n\
 2 1 3  79.1288  30.0000  1.4632    0.0000  0.2065    0.0000  2.0000\n\
 1 3 1  80.7352  16.4130  4.9987    0.0000  0.0843    0.0000  1.0137\n\
 1 3 3  85.4436  14.4937  3.9928    0.0000  1.4350   44.5320  1.1348\n\
 3 3 3  89.9282  32.1199  2.7181    0.0000  0.3323   57.6122  1.0000\n\
 1 3 2  82.9640  32.4874  0.8777    0.0000  0.9627    0.0000  1.0010\n\
 2 3 3  85.7838  17.3139  1.9157    0.0000  3.6306    0.0000  2.1858\n\
 2 3 2  84.2527  33.1226  0.6730    0.0000  0.7238    0.0000  2.4348\n\
 1 2 3   0.0000  14.4588  3.1507    0.0000  3.4571    0.0000  1.0149\n\
 3 2 3   0.0000   0.9696  3.6303    0.0000  0.0000    0.0000  1.6987\n\
 2 2 3   0.0000   0.5797  1.9739    0.0000  0.0000    0.0000  2.4494\n\
 1 1 4  89.5412  10.0000  3.1804    0.0000  0.1000    2.0000  2.0000\n\
 3 1 4  75.6717  34.2176  2.4651    0.0000  0.2576    0.0000  1.3080\n\
 4 1 4  56.7950  21.2889  1.1348    0.0000  0.1000    0.0000  2.0000\n\
 2 1 4  57.5770  16.8737  1.6684    0.0000  0.9500    0.0000  1.0100\n\
 1 2 4   0.0000   0.0100  5.6777    0.0000  0.0000    0.0000  1.2703\n\
 1 3 4  77.6218  14.9138  3.5216    0.0000  0.6416    0.0000  1.5611\n\
 3 3 4  87.1336  29.3985  2.1764    0.0000  0.6955    0.0000  1.8914\n\
 4 3 4  70.2689  19.9584  4.2797    0.0000  0.6998    0.0000  1.6913\n\
 2 3 4  73.7577  44.4943  0.5753    0.0000  3.0692    0.0000  1.5996\n\
 1 4 1  81.0255  35.0000  0.7103    0.0000  1.6888    0.0000  1.0100\n\
 1 4 3  90.0000  25.5053  4.4541    0.0000  2.1016    0.0000  1.2203\n\
 1 4 4  67.3194  22.7804  1.8415    0.0000  2.0063    0.0000  1.0100\n\
 3 4 3  74.8496  50.0000  1.6227   -6.6718  3.0000   50.0000  1.0100\n\
 3 4 4  74.6195  47.0693  0.9622   -3.4101  2.1852    0.0000  1.7988\n\
 4 4 4  66.6498  17.4122  7.0441    0.0000  1.1587    0.0000  1.2779\n\
 1 4 2  90.0000  20.9302  1.2522    0.0000  0.6402    0.0000  3.0000\n\
 2 4 3  83.0567  36.8198  1.0207    0.0000  0.8674    0.0000  3.0000\n\
 2 4 4  79.0108  22.2517  3.0204    0.0000  0.4118    0.0000  2.9985\n\
 2 4 2  49.0517  14.1263  7.4919    0.0000  0.1000    0.0000  1.0100\n\
 1 2 4   0.0000   0.0019  6.0000    0.0000  0.0000    0.0000  1.0400\n\
 3 2 4   0.0000   0.0019  6.0000    0.0000  0.0000    0.0000  1.0400\n\
 4 2 4   0.0000   0.0019  6.0000    0.0000  0.0000    0.0000  1.0400\n\
 2 2 4   0.0000   0.0019  6.0000    0.0000  0.0000    0.0000  1.0400\n\
30 ! Nr of torsions. at1;at2;at3;at4;;V1;V2;V3;p(tor1);p(cot1);n\n\
 1 1 1 1  2.0474  32.6719  0.5282  -9.0000  -2.6449  0.0000  0.0000\n\
 1 1 1 2  1.6328  78.4995 -0.1514  -6.9161  -2.9986  0.0000  0.0000\n\
 2 1 1 2  2.4142  78.7025  0.3506  -8.8640  -6.9283  0.0000  0.0000\n\
 1 1 1 3 -0.7104  22.6038  0.5309  -2.0000  -0.6614  0.0000  0.0000\n\
 2 1 1 3  1.9323  52.9368  0.6554  -8.8118  -3.9854  0.0000  0.0000\n\
 3 1 1 3 -1.2500   1.1248 -0.1230  -9.9453  -3.9000  0.0000  0.0000\n\
 1 1 3 1 -0.6848  56.7751 -1.2733  -2.2937  -4.0000  0.0000  0.0000\n\
 1 1 3 2 -1.4557  78.6279  0.9945  -3.2742  -2.4240  0.0000  0.0000\n\
 2 1 3 1  0.6928  78.1546  0.5608  -3.1713  -3.7301  0.0000  0.0000\n\
 2 1 3 2 -1.4343  77.0699  0.9875  -3.4139  -1.4053  0.0000  0.0000\n\
 1 1 3 3  0.5153   2.1584  0.2000  -6.5859  -3.0000  0.0000  0.0000\n\
 2 1 3 3  0.2018  80.0000  0.3778  -2.5000  -2.8750  0.0000  0.0000\n\
 3 1 3 1 -1.9875  79.2591  1.0000  -2.4206  -3.9342  0.0000  0.0000\n\
 3 1 3 2 -1.1000  78.8002 -1.0000  -2.6282  -4.0000  0.0000  0.0000\n\
 3 1 3 3 -1.0000  83.5323  4.3660  -2.6805  -1.2938  0.0000  0.0000\n\
 1 3 3 1  3.4682   0.0781  0.9887  -6.1195  -0.5004  0.0000  0.0000\n\
 1 3 3 2  1.0000  16.5478 -1.0313  -2.0000  -2.6888  0.0000  0.0000\n\
 2 3 3 2  4.0818  -3.2744 -0.9664  -7.1634  -3.0000  0.0000  0.0000\n\
 1 3 3 3  4.2014 -10.0642  1.8690  -2.4805  -2.5000  0.0000  0.0000\n\
 2 3 3 3  1.0000 -10.0500 -1.0000  -2.1946  -0.5300  0.0000  0.0000\n\
 3 3 3 3  1.0000   1.6871  3.0000  -6.2660  -0.5500  0.0000  0.0000 \n\
 4 1 1 4  3.0000  80.0000  2.0000  -2.0000  -1.8773  0.0000  0.0000\n\
 1 1 1 4  1.0676  41.9735 -0.6803  -6.3125  -3.0000  0.0000  0.0000\n\
 2 1 1 4  3.0000  44.9653  1.7235  -3.0352  -1.0000  0.0000  0.0000\n\
 0 1 4 0  1.4015  77.4788  1.0472  -6.9179  -1.7577  0.0000  0.0000\n\
 0 2 4 0 -3.0000   0.1000  0.0200  -2.8105   0.0000  0.0000  0.0000\n\
 0 3 4 0  3.0000  50.0719  0.2740  -8.0000  -1.0000  0.0000  0.0000\n\
 0 4 4 0  0.8759  30.0000 -1.7701  -8.0000  -1.0000  0.0000  0.0000\n\
 0 1 1 0  3.0000  38.1059  2.0000  -3.2272  -2.9827  0.0000  0.0000\n\
 4 1 4 4 -3.0000  40.0000 -1.8678  -7.3019  -1.0000  0.0000  0.0000\n\
4 ! Nr of hydrogen bonds. at1;at2;at3;r(hb);p(hb1);p(hb2);p(hb3\n\
 3 2 3 1.8130 -3.5409 2.3815 21.9463\n\
 3 2 4 1.7753 -5.0000 3.0000  3.0000\n\
 4 2 3 1.3884 -5.0000 3.0000  3.0000\n\
 4 2 4 1.6953 -4.0695 3.0000  3.0000"


import os

label_map = {
    "+x": "px",
    "-x": "nx",
    "+y": "py",
    "-y": "ny",
    "+z": "pz",
    "-z": "nz",
}

def write_forcefield_to_faces(structure_name, surface_normals, ff_text, base_outdir="slab_faces"):
    for normal in surface_normals:
        tag = label_map[normal]
        face_dir = os.path.join(base_outdir, structure_name, tag)
        os.makedirs(face_dir+'/inputs', exist_ok=True)

        ff_path = os.path.join(face_dir+'/inputs', "ffield.reax.chon")
        with open(ff_path, "w", encoding="utf-8") as f:
            f.write(ff_text)

        print(f"Wrote {ff_path}")

In [66]:
surface_normals = ["+x", "-x", "+y", "-y", "+z", "-z"]

write_forcefield_to_faces(
    structure_name=structure_name,
    surface_normals=surface_normals,
    ff_text=ff_CHNO,
    base_outdir="slab_faces"
)

Wrote slab_faces/C700_000/px/inputs/ffield.reax.chon
Wrote slab_faces/C700_000/nx/inputs/ffield.reax.chon
Wrote slab_faces/C700_000/py/inputs/ffield.reax.chon
Wrote slab_faces/C700_000/ny/inputs/ffield.reax.chon
Wrote slab_faces/C700_000/pz/inputs/ffield.reax.chon
Wrote slab_faces/C700_000/nz/inputs/ffield.reax.chon


In [67]:
lmp_control = "tabulate_long_range     10000 ! denotes the granularity of long range tabulation, 0 means no tabulation\n\n\
nbrhood_cutoff          4.5  ! near neighbors cutoff for bond calculations in A\n\
hbond_cutoff            6.0  ! cutoff distance for hydrogen bond interactions\n\
bond_graph_cutoff       0.3  ! bond strength cutoff for bond graphs\n\
thb_cutoff              0.001 ! cutoff value for three body interactions"
# file = open(os.path.join(path,"lmp_control"),"w")
# file.write(lmp_control)
# file.close()

In [68]:
def write_lmp_control_to_faces(structure_name, surface_normals, lmp_control_text,
                               base_outdir="slab_faces"):
    for normal in surface_normals:
        tag = label_map[normal]
        face_dir = os.path.join(base_outdir, structure_name, tag)
        os.makedirs(face_dir +'/inputs', exist_ok=True)

        ctrl_path = os.path.join(face_dir+'/inputs', "lmp_control")
        with open(ctrl_path, "w", encoding="utf-8") as f:
            f.write(lmp_control_text)

        print(f"Wrote {ctrl_path}")

In [69]:
write_lmp_control_to_faces(
    structure_name=structure_name,
    surface_normals=surface_normals,
    lmp_control_text=lmp_control
)

Wrote slab_faces/C700_000/px/inputs/lmp_control
Wrote slab_faces/C700_000/nx/inputs/lmp_control
Wrote slab_faces/C700_000/py/inputs/lmp_control
Wrote slab_faces/C700_000/ny/inputs/lmp_control
Wrote slab_faces/C700_000/pz/inputs/lmp_control
Wrote slab_faces/C700_000/nz/inputs/lmp_control


# GPU enabled build

In [70]:
import os

label_map = {
    "+x": "px",
    "-x": "nx",
    "+y": "py",
    "-y": "ny",
    "+z": "pz",
    "-z": "nz",
}

def make_lammps_gpu_sh(structure_name, face_tag):
    return f"""#!/bin/bash
#SBATCH -J {structure_name}_{face_tag}
#SBATCH -p gpu
#SBATCH --nodes=1
#SBATCH --gpus=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --time=48:00:00
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user=aringsby@stanford.edu
#SBATCH --export=ALL
#SBATCH -o /home/groups/kmaher/aringsby/slab_anneal/{structure_name}/{face_tag}/logs/%x.%j.out
#SBATCH -e /home/groups/kmaher/aringsby/slab_anneal/{structure_name}/{face_tag}/logs/%x.%j.err
#SBATCH --signal=B:TERM@60

set -euo pipefail

########## config via env ##########
PROJECT="${{PROJECT:-{structure_name}}}"
FACE="${{FACE:-{face_tag}}}"
RUN_ID="${{RUN_ID:-$(date +%Y%m%d-%H%M%S)}}"
LAMMPS_OPTS="${{LAMMPS_OPTS:-}}"
NGPU="${{NGPU:-1}}"

########## paths ##########
PROOT="$GROUP_HOME/aringsby/slab_anneal/$PROJECT/$FACE"
INPUTS="$PROOT/inputs"
OUTROOT="$PROOT/results"
LATEST="$PROOT/latest"
LOGROOT="$PROOT/logs"

mkdir -p "$OUTROOT" "$LOGROOT" "$LATEST"

########## helper: load module stack ##########
load_stack() {{
  module --force purge
  module load devel
  module load "$1"
  module load "$2"
  module load "$3"
}}

########## detect GPU + choose binary/runtime ##########
GPU_CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n1 | tr -d '[:space:]')
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n1)

case "$GPU_CC" in
  6.0|6.*)
    load_stack gcc/10.1.0 openmpi/4.1.2 cuda/12.4.0
    LMP="$HOME/.local/lammps-kokkos-cuda-pascal60/bin/lmp"
    KOKKOS_THREADS="${{SLURM_CPUS_PER_TASK:-8}}"
    ;;

  7.0|7.*)
    load_stack gcc/10.1.0 openmpi/4.1.2 cuda/12.4.0
    LMP="$HOME/.local/lammps-kokkos-cuda-v100/bin/lmp"
    KOKKOS_THREADS="${{SLURM_CPUS_PER_TASK:-8}}"
    ;;

  8.6|8.*)
    load_stack gcc/10.1.0 openmpi/4.1.2 cuda/11.5.0
    LMP="$HOME/.local/lammps-kokkos-cuda-ampere86/bin/lmp"
    KOKKOS_THREADS=1
    ;;

  9.0|9.*)
    load_stack gcc/12.4.0 openmpi/5.0.5 cuda/12.6.1
    LMP="$HOME/.local/lammps-kokkos-cuda-hopper90/bin/lmp"
    KOKKOS_THREADS=1
    ;;

  *)
    echo "ERROR: Unknown compute capability: $GPU_CC"
    exit 2
    ;;
esac

[[ -x "$LMP" ]] || {{ echo "ERROR: LAMMPS binary not found: $LMP"; exit 127; }}

echo "[INFO] GPU compute capability: $GPU_CC"
echo "[INFO] GPU name: $GPU_NAME"
echo "[INFO] Using LMP: $LMP"

export OMP_NUM_THREADS="$KOKKOS_THREADS"
export OMP_PROC_BIND=spread
export OMP_PLACES=threads

########## scratch workdir ##########
RUNROOT="${{SCRATCH:-/scratch}}/lammps/slab_anneal"
workdir="$RUNROOT/${{PROJECT}}/${{FACE}}"
mkdir -p "$workdir"

rsync -a --exclude 'slurm*.out' --exclude 'slurm*.err' "$INPUTS/" "$workdir/"
cd "$workdir"

########## termination handler ##########
on_term() {{
  echo "[WARN] SIGTERM received; syncing outputs back to HOME"
  rsync -avh \\
    --include='lammps_run.log' \\
    --include='log.*' \\
    --include='*.data' \\
    --include='*.restart' \\
    --include='*.restart.*' \\
    --exclude='*' \\
    ./ "$OUTROOT/" || true
}}
trap on_term TERM

########## input ##########
INFILE="slab_anneal_{face_tag}.in"

[[ -f "$INFILE" ]] || {{ echo "ERROR: input not found: $INFILE"; exit 2; }}

echo "[INFO] PROJECT=$PROJECT FACE=$FACE RUN_ID=$RUN_ID"
echo "[INFO] Workdir: $workdir"
echo "[INFO] Node: $(hostname)"
echo "[INFO] CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES:-<unset>}}"
echo "[INFO] GPU(s):"
nvidia-smi -L || true
echo "[INFO] Using input: $INFILE"
echo "[INFO] Starting LAMMPS: $LMP -in $INFILE $LAMMPS_OPTS"
echo "[INFO] OMP_NUM_THREADS=$OMP_NUM_THREADS NGPU=$NGPU"

########## run ##########
srun --export=ALL --cpu-bind=cores --hint=nomultithread \\
  "$LMP" -k on g "$NGPU" t "$OMP_NUM_THREADS" \\
  -pk kokkos neigh half \\
  -sf kk -in "$INFILE" $LAMMPS_OPTS

########## save outputs ##########
rsync -avh \\
  --include='lammps_run.log' \\
  --include='*last.restart' \\
  --include='log.*' \\
  --include='*.data' \\
  --include='species*.out' \\
  --include='species*.out' \\
  --include='bonds*.reaxff' \\
  --exclude='*' \\
  ./ "$OUTROOT/"
"""

def write_face_job_scripts(structure_name, surface_normals, base_outdir="slab_faces"):
    for normal in surface_normals:
        tag = label_map[normal]

        # face directory that already contains slab_anneal_<tag>.in and slab_<tag>.data
        face_dir = os.path.join(base_outdir, structure_name, tag)
        os.makedirs(face_dir, exist_ok=True)

        script_text = make_lammps_gpu_sh(structure_name, tag)
        script_path = os.path.join(face_dir, "lammps_gpu.sh")

        with open(script_path, "w", encoding="utf-8") as f:
            f.write(script_text)

        os.chmod(script_path, 0o755)
        print(f"Wrote {script_path}")

In [71]:
surface_normals = ["+x", "-x", "+y", "-y", "+z", "-z"]

write_face_job_scripts(
    structure_name=structure_name,
    surface_normals=surface_normals,
    base_outdir="slab_faces"
)

Wrote slab_faces/C700_000/px/lammps_gpu.sh
Wrote slab_faces/C700_000/nx/lammps_gpu.sh
Wrote slab_faces/C700_000/py/lammps_gpu.sh
Wrote slab_faces/C700_000/ny/lammps_gpu.sh
Wrote slab_faces/C700_000/pz/lammps_gpu.sh
Wrote slab_faces/C700_000/nz/lammps_gpu.sh


In [72]:
base_outdir

'slab_faces'